#### get text content from data
If you use this dataset, please cite the following paper:
Urchs et al. (2025): taz2024full: Analysing German Newspapers for Gender Bias and Discrimination across Decades, Findings of ACL 2025.

format:
- meta data:

    - "published_on": publishing date in the format YYYY-MM-DDThh:mm:ss+01:00

    - "contains_actors": boolean indicating whether person entities were detected

    - "crawled_on": crawling date

    - "language": always "de"

    - "type": always "article"

    - "author": article author

    - "keywords": topic keywords

    - "token_count": number of tokens


- text:

    - "title": article title

    - "teaser": short summary

    - "text": main body of the article

In [36]:
import pandas as pd
import json
from pprint import pprint
import os
import re
import string

import spacy
nlp = spacy.load('de_core_news_lg') # oder de_core_news_sm

def flatten(xss):
    return [x for xs in xss for x in xs]

In [7]:
c_directory = r"corpus" 

# Iterate over files in directory
dict_year = {}
for name in os.listdir(c_directory):
    dir = os.path.join(c_directory, name)
    for n in os.listdir(dir):
        with open(os.path.join(dir, n)) as f:
            data = json.load(f)
            dict_year[name] = data
print(dict_year.keys())

dict_keys(['2013', '2014', '2022', '2024', '2023', '2015', '2012', '1988', '1986', '1987', '1980', '1989', '1993', '1994', '1995', '1992', '2008', '2001', '2006', '2007', '2000', '2009', '2017', '2010', '2019', '2021', '2020', '2018', '2011', '2016', '1985', '1983', '1984', '1997', '1990', '1999', '1998', '1991', '1996', '2005', '2002', '2003', '2004'])


In [8]:
# separate sentencesy with spacy
def separate_sentences(sents):
    doc = nlp(sents)
    res = []
    for sent in doc.sents:
     res.append(sent)
    return res

In [9]:
# keep text content and id only
def save_txt_to_file():
    text_dir = "corpus_texts/"
    # Create the directory
    try:
        os.mkdir(text_dir)
        print(f"Directory '{text_dir}' created successfully.")
    except FileExistsError:
        print(f"Directory '{text_dir}' already exists.")

    #fill dir
    for key in dict_year.keys():
        #text_dict = {}
        for k in dict_year[key].keys():
            text = dict_year[key][k]['text']['text']
            #text_dicts.append(text_dict)
            with open(f"{text_dir}_{k}.txt", "w") as f:
                    f.write(str(separate_sentences(str(text))))

#save_txt_to_file()

#### get sentences with occurences of AI related words
Notes: 
- Englisch and German for certain terms?
- Specific product names
- ML/AI system terminology
- Nouns only
- Abreviations, too



steps:
- build AI vocabulary
- filter by sentence containing at least one AI vocab
- multiply sentences if they contain multiple terms 

In [10]:
# get AI-related words, sorted according to 'Häufigkeitsklassen'
# https://corpora.uni-leipzig.de/de/res?corpusId=deu_news_2024&word=KI
ki_terms = ['KI','Künstliche Intelligenz', 'ChatGPT', 'AI', 'Copilot', 'Gemini', 'Chatbot','LLM', 'Chat-GPT',
            'Sprachmodell', 'KI-Modell', 'KI-System', 'KI-Tool', 'Artificial Intelligence', 'KI-Anwendung']
ki_terms = [x.lower for x in ki_terms]

#### Sätze säubern, filtern
- Satzzeichen
- (Html, etc)
- POS (use spacy bc TIGER corpus trained)
- Lexikon für Verb- / Adjektivtypen

In [11]:
# preprocessing auxiliary functions

stopwords = nlp.Defaults.stop_words
with open(f"/Users/sofiaromero/Desktop/Thesis/AnthroAI/config_files/spacy_stopwords.txt", "w") as f:
        f.write(str(stopwords))

# this one is not needed if working on Sentence_spacy
def remove_punctuation(corpus):
    cleaned_corpus = corpus.translate(str.maketrans('', '', string.punctuation))
    return(cleaned_corpus)

# remove stop words from text
def remove_stop_words_from_text(cleaned_corpus):
    out = [word for word in cleaned_corpus.lower().split() if not word in stopwords]
    out_text = ' '.join(out)
    return out_text  # , out


def remove_stop_words(token_list):
    try:
        out = [token for token in token_list if not token.is_stop]
        return out
    except:
        return None

def determine_negation(doc):
    negation_tokens = [tok for tok in doc if tok.dep_ == 'neg']
    negation_head_tokens = [token.head for token in negation_tokens]
    for token in negation_head_tokens:
        return [child for child in token.children]


##### get sentences with ki_terms

In [ ]:
# filter ki_terms
c_txt_dir = r"corpus_texts" 
ki_sentences = {}
for dir in os.listdir(c_txt_dir):
    res_tmp = []
    with open(os.path.join(c_txt_dir,dir)) as f:
        sent = f.read()
        sent_l = sent.lower().split(',') # could also account for lemmata, etc
        if any(ki_terms) in sent_l:
            res = remove_punctuation(sent.text)
            print("1",res)
            res_tmp.append(sent.text)
            # add to dict
            ki_sentences [dir] = flatten (res_tmp)

# save to file
filtered_sentences_json = json.dumps(ki_sentences)

pprint(len(filtered_sentences_json))
pprint(filtered_sentences_json)

2013


FileNotFoundError: [Errno 2] No such file or directory: 'corpus_texts/2013'

In [ ]:
#save dict to file
with open("code/generated_data/KI_sentences", "w") as f:
        f.write(str(filtered_sentences_json))


### Daten veranschaulichen:
- Häufigkeit der KI Terme
- Häufigste Wörter
- length of sentences / number of clauses
- number / type of verbs
- number / type of adjektives

In [ ]:
plots_dir = "plots/"